In [1]:
# ─── Clientes oficiais do Google Cloud ───────────────────────────────────────────
from google.cloud import bigquery
from google.cloud import storage

# ─── Plugins personalizados ───────────────────────────────────────────────────────
# from plugins.auxi.cnpj_functions import (convert_date, transforms, remove_table, get_path_files)
# from plugins.operators.unzip import UnzipGCS



# ─── Bibliotecas de terceiros ─────────────────────────────────────────────────────
import pandas as pd
import pandas_gbq as bq
import numpy as np
import requests
import pendulum
from kubernetes.client import models as k8s  


# ─── Biblioteca padrão do Python ─────────────────────────────────────────────────
from datetime import datetime, timedelta
import logging
import os
import urllib.parse
from pathlib import Path
import requests
import re


## COLUNAS E LINKS

In [3]:
colunas_estabelecimento = ['CNPJ_BASICO',
                        'CNPJ_ORDEM',
                        'CNPJ_DV',
                        'MATRIZ_FILIAL',
                        'NOME_FANTASIA',
                        'SITUACAO_CADASTRAL',
                        'DATA_SITUACAO_CADASTRAL',
                        'MOTIVO_SITUACAO',
                        'NOME_CIDADE_EXT',
                        'PAIS',
                        'Data_Inicio_Atividade',
                        'CNAE_PRINCIPAL',
                        'CNAE_SECUNDARIA',
                        'TIPO_LOGRADDOURO',
                        'LOGRADOURO',
                        'NUM',
                        'COMPLEMENTO',
                        'BAIRRO',
                        'CEP',
                        'UF',
                        'MUNICIPIO',
                        'DDD1',
                        'TEL1',
                        'DDD2',
                        'TEL2',
                        'DDD_FAX',
                        'TEL_FAX',
                        'E_MAIL',
                        'SITUACAO_ESPECIAL',
                        'DATA_SIT_ESPECIAL'
                        ]

colunas_empresa = [
                'CNPJ_BASICO',
                'RAZAO_SOCIAL',
                'COD_NATUREZA_JUR',
                'QUALIFICACAO_RESPONSAVEL',
                'CAPITAL_SOCIAL',
                'COD_PORTE_EMPRESA',
                'ENTE_RESPONSAVEL'
            ]

colunas_simples = ['CNPJ_BASICO', 
                   'OPCAO_PELO_SIMPLES', 
                   'DATA_OPCAO_SIMPLES', 
                   'DATA_EXCLUSAO_DO_SIMPLES',
                   'OPCAO_PELO_MEI',
                   'DATA_OPCAO_MEI',
                   'DATA_EXCLUSAO_DO_MEI']

colunas_socio = ['CNPJ_BASICO'
        ,'COD_TIPO_SOCIO'
        ,'NOME_SOCIO'
        ,'CNPJ_CPF_SOCIO'
        ,'COD_QUALI_SOCIO'
        ,'DATA_ENT_SOCIEDADE'
        ,'PAIS'
        ,'REPRESENTANTE_LEGAL_CPF'
        ,'NOME_REPRESENTANTE'
        ,'COD_QUALI_REPRESENTANTE'
        ,'COD_FAIXA_ETARIA'
        ]

In [4]:
##LINK 
download_files = {
'empresas':  [f'Empresas{i}.zip' for i in range(10)],
'estabelecimentos': [f'Estabelecimentos{i}.zip' for i in range(10)],
'socios': [f'Socios{i}.zip' for i in range(10)],
'dimensoes': ['Cnaes.zip', 'Municipios.zip', 'Naturezas.zip', 'Paises.zip', 'Qualificacoes.zip','Simples.zip'],
'regimes': ['https://arquivos.receitafederal.gov.br/public.php/dav/files/MPPfFit7g7zdA8C/entidades-lucro-real.zip',
'https://arquivos.receitafederal.gov.br/public.php/dav/files/MPPfFit7g7zdA8C/entidades-lucro-presumido.zip',
'https://arquivos.receitafederal.gov.br/public.php/dav/files/MPPfFit7g7zdA8C/entidades-lucro-arbitrado.zip'
]
}

# PERIODO DE APURAÇÃO

In [5]:
##DEFINIÇÃO DO PERÍODO DE APURAÇÃO
def periodo_apuracao_PA():
    pa = datetime.now().strftime("%Y-%m")
    # logging.info(f"Período de apuração definido como: {pa}")
    return pa
periodo_apuracao_PA()

'2026-04'

In [6]:

def check_site_url(pa: str, min_expected_files: int = 37, **context):
    webdav_url = f"https://arquivos.receitafederal.gov.br/public.php/dav/files/YggdBLfdninEJX9/{pa}/"

    resp = requests.request("PROPFIND", webdav_url, headers={"Depth": "1"}, timeout=30, verify=False)
    resp.raise_for_status()
    qtd_arquivos = len(re.findall(r"<(?:d|D):response", resp.text)) - 1
    if qtd_arquivos < 10:
        print(f"Dados disponíveis para {pa}: {qtd_arquivos} arquivos.")
    return True


In [7]:
check_site_url(pa=periodo_apuracao_PA())

c:\Users\Nelio\anaconda3\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'arquivos.receitafederal.gov.br'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


True

# DOWNLOAD

In [23]:
def task_download_cnpj(PA:str, categoria:str, arquivo:str):

    url = f"https://arquivos.receitafederal.gov.br/public.php/dav/files/YggdBLfdninEJX9/{PA}/{arquivo}"
    print(f'URL: {url}')
    # caminho = f"raw/cnpj/{PA}/{categoria}/{arquivo}"
    caminho = rf"C:\Users\Nelio\Downloads\dadosCnpj2026\{PA}\{categoria}\{arquivo}"
    os.makedirs(os.path.dirname(caminho), exist_ok=True)
    response = requests.get(url, stream=True, timeout=300, verify=False)
    response.raise_for_status()
    with open(caminho, "wb") as file:
        for chunk in response.iter_content(chunk_size=1024*1024):
            file.write(chunk)
    # client= storage.Client(credentials=credentials)
    # logging.info(f'Cliente: {client}')
    # logging.info(f'Bucket: {bucket_name}')
    # bucket = client.bucket(bucket_name)
    # blob = bucket.blob(caminho)
    # blob.upload_from_filename(arquivo)
    print(f'Arquivo {arquivo} enviado com sucesso para o GCS em {caminho}.')

### EXECUÇÕES

In [14]:
pa = periodo_apuracao_PA()
pa

'2026-04'

In [2]:
check_site_url(pa=periodo_apuracao_PA())

NameError: name 'check_site_url' is not defined

In [8]:
dimensoes = task_download_cnpj(PA=pa, categoria='dimensoes',arquivo=download_files['estabelecimentos'])


NameError: name 'task_download_cnpj' is not defined